# Summary 

### References

* https://github.com/llnl/Wintap-Analytics/tree/main/2025-acme4-explore
* https://gdo168.llnl.gov/
* https://gdo168.llnl.gov/data/newdocs/datadict/

Download:
https://gdo168.llnl.gov/data/ACME4/stdview-20240819-20240923/process_file.parquet

In [ ]:
%%time
import io
import logging as lg
import numpy as np
import os
import pandas as pd
import re
from collections import Counter
import igraph as ig

# Modelling
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay, roc_curve, auc

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

## our matching algorithm
import sys
import os
sys.path.insert(0,os.path.abspath('../Matching'))
import igraph_io as igio
import fast_match as fm  ## fast version

# Exploring the files


In [ ]:
## the table can be found here: https://gdo168.llnl.gov/data/ACME4/stdview-20240819-20240923/
files = pd.read_parquet("Data/process_file.parquet")


In [ ]:
files.columns

In [ ]:
files[['pid_hash', 'process_name', 'filename']].head(10)

In [ ]:
## all filepaths
Files = files.filename.tolist()
print(len(Files),'filepaths')
## unique filepaths
Files = list(set(Files))
print(len(Files),'unique filepaths')


In [ ]:
## split, look at first element
S = [f.split('\\') for f in Files]
print('first item:',Counter([s[0] for s in S]))

## drop first element
S = [s[1:] for s in S]
Counter([len(s) for s in S])


# (1) Examples of matches given a specific path

There are many distinct paths, and many partial matches between paths.

To limit our search, we consider
* paths of a given length only
* 1-off matches only
* mismatches in last item are just counted as those are very frequent
* we return only the first internal mismatch are report the count
 

In [ ]:
%%time
TreeData = []
path_len = 16 ## fix some length, we tried several
users = []
for s in S:
    # if len(s) > path_len:
    if len(s) == path_len:
        sg = ig.Graph.TupleList([(i,i+1) for i in range(len(s)-1)], directed=True)
        sg.vs['dir'] = s
        if '' not in s:
            TreeData.append(igio.igraph_to_treedata(sg, phi_name='dir'))
## encode all trees
fast = fm.FastTreePathMatcher()
fast.fit_encoder(TreeData)
enc = [fast.encode_tree(t) for t in TreeData]
len(enc)

In [ ]:
for _rep in range(3):
    ctr = 0
    ctr_last = 0
    base = np.random.choice(len(enc)) ## pick one at random then compare with the rest
    l = len(enc[base].label_ids)
    for i in range(len(enc)):
        if (i!=base) and (len(enc[i].label_ids)==l):
            _, score = fast.predict_encoded(enc[base], enc[i])
            if score==(l-1): 
                if _[l-2] != (l-2,l-2):
                    ctr += 1
                    if ctr==1: ## only print one hit
                        print(enc[base].tree.label)
                        print(enc[i].tree.label) 
                else:
                    ctr_last+=1
    if ctr>0:
        print(ctr,'mismatch inside path',ctr_last,'mismatch on last item\n')


# (2) users as templates


In [ ]:
Users = {'baduser3','baduser9','ghostuser1','ghostuser2','user1','user10','user11','user2','user20','user3','user4','user6','user8','user9'}


In [ ]:
label = 'baduser9'
L = []

#for path_len in [3,4,5,6,7,8,9,10,11,12,13,14,15,16]: ## slow
for path_len in [3,4,5,6]: ## sample
    print(path_len)
    TreeData = []
    users = []
    for s in S:
        if len(s) == path_len:
            sg = ig.Graph.TupleList([(i,i+1) for i in range(len(s)-1)], directed=True)
            sg.vs['dir'] = s
            if len(set(s).intersection(Users))>0:
                u = set(s).intersection(Users).pop()
                TreeData.append(igio.igraph_to_treedata(sg, phi_name='dir'))
                users.append(u)
    
    freq = dict(Counter(users))
    
    ## encode all trees
    fast = fm.FastTreePathMatcher()
    fast.fit_encoder(TreeData)
    enc = [fast.encode_tree(t) for t in TreeData]
    len(enc)

    counts1 = {x:0 for x in Users}
    counts2 = {x:0 for x in Users}
    templates = np.where(np.array(users)==label)[0].tolist()
    for i in range(len(enc)):
        s = enc[i].tree.label
        if (label not in s):
            max_score = 0
            for base in templates:
                _, score = fast.predict_encoded(enc[base], enc[i])
                if score>max_score:
                    max_score=score
            if max_score==(path_len-1):
                u = set(s).intersection(Users).pop()
                counts1[u] += 1
            elif max_score==(path_len-2):
                u = set(s).intersection(Users).pop()
                counts2[u] += 1
    
    for i in counts1.keys():
        if i != label:
            L.append([i,path_len,counts1[i]/freq[i],counts2[i]/freq[i]])
df_bad = pd.DataFrame(L, columns=['user','len','1-off','2-off'])
df_bad['1 or 2 off'] = df_bad['1-off'] + df_bad['2-off']


In [ ]:
# 1. Pivot the dataframe so X and Y are your axes, and Intensity is the value
pivot_df_bad = df_bad.pivot(index="user", columns="len", values="1-off")

# 2. Create the levelplot/heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(pivot_df_bad, cmap="viridis", annot=True, fmt=".2f") # Adjust fmt based on your numbers

# 3. Add labels and display
plt.xlabel("Path length")
plt.ylabel("User")
plt.title("Proportion of 1-off matches vs baduser9")
plt.show()


# (3) Pick a redteam PID with several files

Find other non-redteam PIDs with several hits


In [ ]:
process = pd.read_parquet("Data/process_uber_summary.parquet")
process.shape


In [ ]:
red = process[['pid_hash','red_team']]
#red[red['red_team']==1]
red = red.set_index('pid_hash')['red_team'].to_dict()


In [ ]:
## example with baduser3
badfiles = files[files['pid_hash'] == 'BB8D51837C4CD278B09A83D92341D00B']['filename'].tolist()
print(badfiles[0])
len(badfiles)


In [ ]:
## badfiles -- use as templates
_ = [f.split('\\') for f in badfiles]
print('first item:',Counter([s[0] for s in _]))
badfiles = [s[1:] for s in _ if len(s)>2]
Counter([len(s) for s in badfiles])


In [ ]:
## pick candidates with similar number of files
_ctr = Counter(files['pid_hash'])
candidates = [str(k) for k, c in _ctr.items() if 55 <= c <= 65]
len(candidates)


In [ ]:
candidate_paths = []
for c in candidates:
    if red.get(c,1) == 0: ## pick non redteam
        c_files = files[files['pid_hash'] == c]['filename'].tolist()
        _ = [f.split('\\') for f in c_files]
        c_files = [s[1:] for s in _ if (len(s)>2) and (len(s)<12)] ## also limit size
        candidate_paths.append( (c,c_files) )  

In [ ]:
TreeData = []
for s in badfiles:
    sg = ig.Graph.TupleList([(i,i+1) for i in range(len(s)-1)], directed=True)
    sg.vs['dir'] = s
    TreeData.append(igio.igraph_to_treedata(sg, phi_name='dir'))
for (h,c) in candidate_paths:
    for s in c:
        sg = ig.Graph.TupleList([(i,i+1) for i in range(len(s)-1)], directed=True)
        sg.vs['dir'] = s
        TreeData.append(igio.igraph_to_treedata(sg, phi_name='dir'))    
## encode all trees
fast = fm.FastTreePathMatcher()
fast.fit_encoder(TreeData)
enc = [fast.encode_tree(t) for t in TreeData]
len(enc)


In [ ]:
L = []
l = len(badfiles)
lens = [len(x[1]) for x in candidate_paths]
for i in range(len(badfiles)):
    gap = l
    for c in range(len(candidate_paths)):
        max_score = 0
        for j in range(lens[c]):
            _, score = fast.predict_encoded(enc[i], enc[j+gap])
            if score>max_score:
                max_score = score
        gap += lens[c]
        if max_score>=3:
            L.append([i,c,int(max_score),len(badfiles[i])])
df_pid = pd.DataFrame(L, columns=['badfile','candidate','score','max_score'])
df_pid['ratio'] = df_pid['score'] / df_pid['max_score']
df_pid.sort_values(by='score', ascending=False)


In [ ]:
_ = df_pid.groupby(by='candidate').sum().sort_values(by='ratio', ascending=False)
top_candidates = _.nlargest(25,'ratio').index.tolist()
_df = df_pid[df_pid.candidate.isin(top_candidates)]


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Pivot the dataframe so X and Y are your axes, and Intensity is the value
pivot_df = _df.pivot(index="candidate", columns="badfile", values="ratio")
_s = _df.groupby(by='candidate').sum()['ratio'].sort_values(ascending=False).index.tolist()
pivot_df = pivot_df.reindex(_s)
# 2. Create the levelplot/heatmap
plt.figure(figsize=(14, 7))
sns.heatmap(pivot_df, cmap="Blues", annot=False)

# 3. Add labels and display
plt.xlabel("Template filepath index")
plt.ylabel("Candidate process index")
plt.title("Longest matching (sub)paths")
plt.show()


In [ ]:
l = [candidates[i] for i in top_candidates[:4]]+['BB8D51837C4CD278B09A83D92341D00B']
process[process['pid_hash'].isin(l)][['pid_hash','red_team','user_name','process_name']]
    